# Nail Retouch Paired Edit v1

This notebook prepares a paired `before -> after` manicure retouch dataset and trains a first-pass **pix2pix-turbo** model on it.

It is designed for structure-preserving retouch, not style-only target-image learning.

## 1. Clone Project

Use this in a fresh Colab runtime:
```bash
cd /content && rm -rf nail-retouch-assistant && git clone https://github.com/Zifpen/nail-retouch-assistant.git
```

Expected Drive dataset path:
- `/content/drive/MyDrive/paired_edit_strict_plus/train_A`
- `/content/drive/MyDrive/paired_edit_strict_plus/train_B`
- `/content/drive/MyDrive/paired_edit_strict_plus/test_A`
- `/content/drive/MyDrive/paired_edit_strict_plus/test_B`


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import yaml

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/nail-retouch-assistant')
CONFIG_PATH = PROJECT_ROOT / 'colab' / 'paired_edit_config.yaml'
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))

DRIVE_DATASET_DIR = Path(cfg['drive_dataset_dir'])
LOCAL_DATASET_DIR = Path(cfg['project_root']) / cfg['dataset_dir']
REPO_DIR = Path(cfg['repo_dir'])

if not DRIVE_DATASET_DIR.exists():
    raise FileNotFoundError(f'Missing dataset in Drive: {DRIVE_DATASET_DIR}')

for rel in ('train_A', 'train_B', 'test_A', 'test_B', 'train_prompts.json', 'test_prompts.json'):
    if not (DRIVE_DATASET_DIR / rel).exists():
        raise FileNotFoundError(f'Missing paired-edit dataset file: {DRIVE_DATASET_DIR / rel}')

shutil.rmtree(LOCAL_DATASET_DIR, ignore_errors=True)
LOCAL_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    f"cp -a '{DRIVE_DATASET_DIR}' '{LOCAL_DATASET_DIR}'",
    shell=True,
    check=True,
    executable='/bin/bash',
)
print(f'Copied dataset to {LOCAL_DATASET_DIR}')

if not REPO_DIR.exists():
    subprocess.run(
        f"git clone {cfg['repo_url']} {REPO_DIR}",
        shell=True,
        check=True,
    )

subprocess.run(f"python -m pip install -q pyyaml", shell=True, check=True)
subprocess.run(f"python -m pip install -q -r {REPO_DIR / 'requirements.txt'}", shell=True, check=True)
subprocess.run("python -m pip install -q vision-aided-loss clean-fid", shell=True, check=True)
print('setup complete')


In [ ]:
from pathlib import Path
import json
import torch
import yaml

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
dataset_dir = Path(cfg['project_root']) / cfg['dataset_dir']
output_dir = Path(cfg['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. Switch Colab runtime to T4/L4/A100 before training.')
print('GPU:', torch.cuda.get_device_name(0))
print('dataset:', dataset_dir)
cfg


In [ ]:
import json
from pathlib import Path

train_prompts = json.loads((dataset_dir / 'train_prompts.json').read_text(encoding='utf-8'))
test_prompts = json.loads((dataset_dir / 'test_prompts.json').read_text(encoding='utf-8'))
print('train pairs:', len(train_prompts))
print('test pairs:', len(test_prompts))
print('example prompt:', next(iter(train_prompts.values())))


In [ ]:
import shlex

train_cfg = cfg['training']
cmd_parts = [
    'cd',
    shlex.quote(cfg['repo_dir']),
    '&&',
    'accelerate',
    'launch',
    'src/train_pix2pix_turbo.py',
    f"--pretrained_model_name_or_path={shlex.quote(cfg['pretrained_model_name_or_path'])}",
    f"--output_dir={shlex.quote(cfg['output_dir'])}",
    f"--dataset_folder={shlex.quote(str(dataset_dir))}",
    f"--resolution={train_cfg['resolution']}",
    f"--train_batch_size={train_cfg['train_batch_size']}",
    f"--num_training_epochs={train_cfg['num_training_epochs']}",
    f"--viz_freq={train_cfg['viz_freq']}",
]

if train_cfg.get('enable_xformers_memory_efficient_attention', False):
    cmd_parts.append('--enable_xformers_memory_efficient_attention')
if train_cfg.get('track_val_fid', False):
    cmd_parts.append('--track_val_fid')

cmd = ' '.join(cmd_parts)
print(cmd)


In [ ]:
import os
import subprocess

env = os.environ.copy()
env['WANDB_MODE'] = 'disabled'
env['WANDB_DISABLED'] = 'true'

process = subprocess.Popen(
    cmd,
    shell=True,
    executable='/bin/bash',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')

process.wait()
print('\nRETURN CODE:', process.returncode)
if process.returncode != 0:
    raise RuntimeError(f'Paired-edit training failed with exit code {process.returncode}')


In [ ]:
from pathlib import Path
import shutil

DRIVE_OUTPUT_DIR = Path(cfg['drive_output_dir'])
LOCAL_OUTPUT_DIR = Path(cfg['output_dir'])

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for item in LOCAL_OUTPUT_DIR.iterdir():
    dst = DRIVE_OUTPUT_DIR / item.name
    if item.is_dir():
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(item, dst)
    else:
        shutil.copy2(item, dst)

print(f'Copied paired-edit outputs to {DRIVE_OUTPUT_DIR}')


## Notes

- This notebook follows the paired-data `pix2pix-turbo` training format described in the official `img2img-turbo` docs.
- We disable `wandb` by default so training can start without an API key.
- The exported dataset is expected to contain `train_A/train_B/test_A/test_B` plus prompt JSON files.
- After training, validate on the five manicure holdout pairs before changing the dataset again.
